# Metrics

1. Abnormal returns
2. Standardized Unexpected Earnings (SUE)
3. Portfolio Assignment
4. Continuously balanced SUE

### **3.2.1. Abnormal Returns**
We first load CRSP global NYSE data and selected firms data to measure abnormal returns. Abnormal returns are measured as:

$$ \begin{aligned}
&AR_{jt}=R_{jt}-R_{pt} \\
&AR_{jt}=\text{Abnormal return for firm j, day t} \\
&R_{jt}\ \ \ =\text{Raw return for j, day t} \\
&R_{pt}\ \ \ =\text{Equally weighted mean return for day t on the NYSE} \\
&\qquad \ \ \  \ \ \   \text{firm size (market value of common equity) decile that} \\
&\qquad \ \ \  \ \ \ \text{j is a part of at the beginning of the calendar year}
\end{aligned}$$

We use *DlyCap* (in thousand of dollars) as a proxy for firm size and define deciles for each calendar year.

In [37]:
import pandas as pd

# Loading data
dfD_NYSE = pd.read_csv("data/dfD_NYSE.csv")
dfD_NYSE['DlyCalDt'] = pd.to_datetime(dfD_NYSE['DlyCalDt'])
dfD_NYSE['Year'] = dfD_NYSE['DlyCalDt'].dt.year

/var/folders/f_/8m_p_02n2rj7nqhb_056hsbh0000gn/T/ipykernel_16753/1267360022.py:4: DtypeWarning: Columns (0: HdrCUSIP) have mixed types. Specify dtype option on import or set low_memory=False.
  dfD_NYSE = pd.read_csv("data/dfD_NYSE.csv")


We define the first trading day of each year before assigning market cap deciles for each given year.

In [38]:
# First trading day per year
first_trading_day = (
    dfD_NYSE.groupby('Year')['DlyCalDt']
    .min()
    .reset_index()
    .rename(columns={'DlyCalDt': 'FirstTradingDay'})
)

# DlyCap on first trading day for each firm and assign size deciles to each firm for that year
jan_cap = dfD_NYSE.merge(
    first_trading_day[['Year', 'FirstTradingDay']],
    left_on=['Year', 'DlyCalDt'],
    right_on=['Year', 'FirstTradingDay']
)[['PERMNO', 'Year', 'DlyCap']]

jan_cap['SizeDecile'] = jan_cap.groupby('Year')['DlyCap'].transform(
    lambda x: pd.qcut(x, 10, labels=False, duplicates='drop') + 1
)

# Print decile boundaries per year
for year, group in jan_cap.groupby('Year'):
    print(f"\n{year}")
    print(group.groupby('SizeDecile')['DlyCap'].agg(['min', 'max', 'count']))

# Attach size decile to every daily observation (Year + PERMNO)
dfD_NYSE = dfD_NYSE.merge(jan_cap[['PERMNO', 'Year', 'SizeDecile']], on=['PERMNO', 'Year'], how='left')

# Equally weighted mean return per decile per day (R_pt)
decile_returns = (
    dfD_NYSE.groupby(['DlyCalDt', 'SizeDecile'])['DlyReturns']
    .mean()
    .reset_index()
    .rename(columns={'DlyReturns': 'R_decile'})
)


1974
                  min          max  count
SizeDecile                               
1.0            324.19      3434.00    272
2.0           3441.00      6441.94    272
3.0           6462.50     11037.25    272
4.0          11048.00     18036.00    272
5.0          18048.00     28832.00    272
6.0          28887.25     48441.25    271
7.0          48541.50     88256.00    272
8.0          88395.00    184616.50    272
9.0         184800.00    493920.00    272
10.0        494643.75  35250698.50    272

1975
                  min          max  count
SizeDecile                               
1.0            244.13      2275.00    269
2.0           2280.13      4371.75    266
3.0           4374.75      7239.38    267
4.0           7246.75     11496.00    267
5.0          11537.50     20075.00    267
6.0          20142.25     33656.00    267
7.0          33671.88     62029.50    267
8.0          62230.88    131085.00    267
9.0         131717.50    363090.00    267
10.0        365030.50 

In [39]:
dfD = pd.read_csv("data/dfD_final.csv")
dfD['DlyCalDt'] = pd.to_datetime(dfD['DlyCalDt'])
dfD['Year'] = dfD['DlyCalDt'].dt.year

# Assign full-NYSE size decile to each firm in dfD_final
dfD = dfD.merge(jan_cap[['PERMNO', 'Year', 'SizeDecile']], on=['PERMNO', 'Year'], how='left')

# Merge benchmark return (R_pt) and compute abnormal return
dfD = dfD.merge(decile_returns, on=['DlyCalDt', 'SizeDecile'], how='left')
dfD['DlyAbnReturn'] = dfD['DlyReturns'] - dfD['R_decile']

# Check
print(dfD[['PERMNO', 'DlyCalDt', 'DlyReturns', 'SizeDecile', 'R_decile', 'DlyAbnReturn']].head(10))
print(f"\nMissing DlyAbnReturn: {dfD['DlyAbnReturn'].isna().sum()}\nRows: {len(dfD)}")

   PERMNO   DlyCalDt  DlyReturns  SizeDecile  R_decile  DlyAbnReturn
0   10006 1974-01-02         NaN         9.0       NaN           NaN
1   10006 1974-01-03    0.058315         9.0  0.032230      0.026085
2   10006 1974-01-04   -0.024490         9.0  0.005894     -0.030384
3   10006 1974-01-07   -0.025105         9.0 -0.000932     -0.024172
4   10006 1974-01-08   -0.023605         9.0 -0.017037     -0.006569
5   10006 1974-01-09   -0.070330         9.0 -0.027641     -0.042689
6   10006 1974-01-10    0.004728         9.0 -0.010495      0.015224
7   10006 1974-01-11    0.037647         9.0  0.011104      0.026543
8   10006 1974-01-14   -0.013605         9.0  0.001394     -0.014999
9   10006 1974-01-15    0.002299         9.0  0.005657     -0.003358

Missing DlyAbnReturn: 732453
Rows: 6670760


We are missing quite a few DlyAbnReturn. This can likely be explained by the fact that we don't handle cases of firms not present on the NYSE/AMEX on the first trading day of the year

In [40]:
# For firms missing a decile for a given year, use their first available DlyCap in that year instead of earliest Jan
first_cap_per_firm_year = (
    dfD_NYSE[dfD_NYSE['DlyCap'].notna()]
    .sort_values('DlyCalDt')
    .groupby(['PERMNO', 'Year'])
    .first()
    .reset_index()[['PERMNO', 'Year', 'DlyCap']]
)

# Recompute deciles on this expanded table
first_cap_per_firm_year['SizeDecile'] = first_cap_per_firm_year.groupby('Year')['DlyCap'].transform(
    lambda x: pd.qcut(x, 10, labels=False, duplicates='drop') + 1
)

# Use this as fallback for firms missing decile
dfD = dfD.merge(
    first_cap_per_firm_year[['PERMNO', 'Year', 'SizeDecile']].rename(columns={'SizeDecile': 'SizeDecile_fallback'}),
    on=['PERMNO', 'Year'], how='left'
)

dfD['SizeDecile'] = dfD['SizeDecile'].fillna(dfD['SizeDecile_fallback'])
dfD = dfD.drop(columns='SizeDecile_fallback')
dfD = dfD[dfD['SizeDecile'].notna()] # 1012 rows that don't have any DlyCap over the year

# Recompute AbnReturns with filled deciles
dfD = dfD.merge(decile_returns, on=['DlyCalDt', 'SizeDecile'], how='left', suffixes=('_old', ''))
dfD = dfD.drop(columns=[c for c in dfD.columns if c.endswith('_old')])
dfD['DlyAbnReturn'] = dfD['DlyReturns'] - dfD['R_decile']


print(f"NaN DlyAbnReturn after Jan-fix fallback: {dfD['DlyAbnReturn'].isna().sum()} our of {len(dfD)} rows")

NaN DlyAbnReturn after Jan-fix fallback: 611648 our of 6656235 rows


We are able to get back about 103 183 daily returns this way. Remaining missing values come from missing Closing price values.

The paper mentions that "*Observations were excluded from the analysis if the return for the earnings announcement day was missing on CRSP, or if the CRSP returns series did not encompass the 160 trading days surrounding the earnings announcement*". We hence exclude:
1. Quarters for which the return for the earnings announcement day is missing
2. When the CRSP returns series do not respect the 160 trading days window. *I imagine that 160 trading days corresponds to a ±80 days symmetrical window, but they after that they only mention a ±60 days window in the paper. I also interpreted that we only validate 160 trading days with continuous AbnormalReturns*

In [41]:
dfQ_final = pd.read_csv("data/dfQ_final.csv")
dfQ_final['rdq'] = pd.to_datetime(dfQ_final['rdq'])

dfQ_final = dfQ_final.merge(
    dfD[['PERMNO', 'DlyCalDt', 'DlyAbnReturn']],
    left_on=['PERMNO', 'rdq'],
    right_on=['PERMNO', 'DlyCalDt'],
    how='left'
)

# Exclude if return missing on announcement day
before = len(dfQ_final)
dfQ_final = dfQ_final[dfQ_final['DlyAbnReturn'].notna()]
print(f"Returns on earnings announcement day missing: Dropped {before - len(dfQ_final)} firm-quarters, {len(dfQ_final)} remaining")


Returns on earnings announcement day missing: Dropped 24177 firm-quarters, 90840 remaining


In [42]:
# For each PERMNO, get sorted array of trading days with valid returns
trading_days = (
    dfD[dfD['DlyAbnReturn'].notna()]
    .groupby('PERMNO')['DlyCalDt']
    .apply(lambda x: sorted(x))
    .reset_index()
    .rename(columns={'DlyCalDt': 'TradingDays'})
)

dfQ_final = dfQ_final.merge(trading_days, on='PERMNO', how='left')

# For each firm-quarter, find index of announcement date and check 80 trading days on each side
def check_160_trading_days(row):
    dates = row['TradingDays']
    rdq = row['rdq']
    if rdq not in dates:
        return False
    idx = dates.index(rdq)
    return (idx >= 80) and (idx + 80 < len(dates))

before = len(dfQ_final)
dfQ_final['TradingDays'] = dfQ_final['TradingDays'].apply(list)
mask = dfQ_final.apply(check_160_trading_days, axis=1)
dfQ_final = dfQ_final[mask].drop(columns='TradingDays')
print(f"160-days window missing: Dropped {before - len(dfQ_final)} firm-quarters, {len(dfQ_final)} remaining")

160-days window missing: Dropped 4883 firm-quarters, 85957 remaining


We have 85957 firm-quarters using epspxq. This is slightly above the 84792 used by the authors. We need to check with the other EPS, and with some other cleaning methods.

### **3.2.2. Estimation of standardized unexpected earnings (SUE)**

"*Earnings were forecasted by estimating the Foster (1977) model with historical data. [The Foster model assumes that earnings follow a first-order autoregressive process in seasonal differences. FOS indicate that they used a maximum of 20 observations to estimate the Foster model. We used a maximum of 24 observations. FOS indicate that firms were included in the sample even if only ten consecutive quarters of data were available. We retained such firms also, but where fewer than 16 observations were available, we assumed that earnings followed a seasonal random walk.]*" 

"*The difference between actual and forecasted earnings was then scaled by the standard deviation of forecast errors over the estimation period to obtain standardized unexpected earnings or *SUE*.*"


**Data inclusion critera**
- Maximum of 24 observation to estimate the regression
- If 10 < quarters < 16, we retain firms but assume that earnings followed a seasonal random walk.
- We scale the residual by the Sd of forecast errors over the estimation period


Min: 10 quarters. Max: 24 quarters.

We first load the entire Compustat history, check past consecutive quarters (notably before 1974) for firms shortlisted from the 1974-1986 window. We then precompute Foster regressors.

In [ ]:
# Loading entire Compustat (1961-1987)
dfQ_full = pd.read_csv("data/WRDS/Compustat_Q_196103_198712.csv", low_memory=False)
dfQ_full['datadate'] = pd.to_datetime(dfQ_full['datadate'])

# Checking history for firms selected from the 1974-1986 window
# Filter in int then convert to str
shortlisted_gvkeys_int = set(int(g) for g in dfQ_final['gvkey'].unique())
dfQ_full = dfQ_full[dfQ_full['gvkey'].isin(shortlisted_gvkeys_int)]
dfQ_full['gvkey'] = dfQ_full['gvkey'].astype(str)
print(f"After filter: {dfQ_full.shape}")

dfQ_full = dfQ_full.sort_values(['gvkey', 'fyearq', 'fqtr']).reset_index(drop=True)
dfQ_full = dfQ_full.dropna(subset=['epspxq'])
print(f"After dropna epspxq: {dfQ_full.shape}")

results = []

After filter: (182338, 18)
After dropna epspxq: (177284, 18)

First firm 10000: 87 quarters
Date range: 1966-06-30 00:00:00 to 1987-12-31 00:00:00
    datadate  pidx  epspxq  y_foster  x_foster
0 1966-06-30  7866    0.11       NaN       NaN
1 1966-09-30  7867    0.22       NaN       NaN
2 1966-12-31  7868    0.39       NaN       NaN
3 1967-03-31  7869    0.19       NaN       NaN
4 1967-06-30  7870    0.23      0.12       NaN
5 1967-09-30  7871    0.37      0.15      0.12
6 1967-12-31  7872    0.50      0.11      0.15
7 1968-03-31  7873    0.28      0.09      0.11
8 1968-06-30  7874    0.33      0.10      0.09
9 1968-09-30  7875    0.40      0.03      0.10


To optimize the computation of SUE on all firm-quarters, we pre calculate needed metrics on a global basis.

In [ ]:
# Quarter index
dfQ_full['pidx'] = dfQ_full['fyearq'] * 4 + dfQ_full['fqtr']

# Precompute lags per firm
g = dfQ_full.groupby('gvkey')['epspxq']
dfQ_full['eps_l1'] = g.shift(1)
dfQ_full['eps_l4'] = g.shift(4)
dfQ_full['eps_l5'] = g.shift(5)

# Foster regressors
dfQ_full['y_foster'] = dfQ_full['epspxq'] - dfQ_full['eps_l4']
dfQ_full['x_foster'] = dfQ_full['eps_l1'] - dfQ_full['eps_l5']

# pidx lags for consecutiveness check
dfQ_full['pidx_l1'] = dfQ_full.groupby('gvkey')['pidx'].shift(1)
dfQ_full['pidx_l4'] = dfQ_full.groupby('gvkey')['pidx'].shift(4)
dfQ_full['pidx_l5'] = dfQ_full.groupby('gvkey')['pidx'].shift(5)

# Check first firm
gvkey_0, firm_0 = next(iter(dfQ_full.groupby('gvkey')))
firm_0 = firm_0.reset_index(drop=True)
print(f"\nFirst firm {gvkey_0}: {len(firm_0)} quarters")
print(f"Date range: {firm_0['datadate'].min()} to {firm_0['datadate'].max()}")
print(firm_0[['datadate', 'pidx', 'epspxq', 'y_foster', 'x_foster']].head(10))

We then iterate on each firm, identified with its gvkey. We respect the 10 minimum required quarters, the maximum 24 quarters used to estimate the regression, and the seasonal random walk for firms with 10 to 16 quarters.

In [ ]:
import numpy as np

for gvkey, firm in dfQ_full.groupby('gvkey'):
    firm = firm.reset_index(drop=True)

    for i in range(len(firm)):
        event_row = firm.iloc[i]

        # Only compute SUE for event quarters in 1974-1986
        if not (pd.Timestamp('1974-01-01') <= event_row['datadate'] <= pd.Timestamp('1986-12-31')): continue

        # History: max 24, min 10
        hist = firm.iloc[max(0, i-24):i].copy().reset_index(drop=True)
        if len(hist) < 10: continue

        # Keep only the last consecutive block
        diffs = hist['pidx'].diff()
        breaks = diffs[diffs != 1].index.tolist()
        if breaks:
            hist = hist.iloc[breaks[-1]:].reset_index(drop=True)
        if len(hist) < 10: continue

        # Extract lagged EPS values
        q_last  = hist.iloc[-1]['epspxq']   # Q_{t-1}
        q_last4 = hist.iloc[-4]['epspxq']   # Q_{t-4} (same quarter last year)
        q_last5 = hist.iloc[-5]['epspxq'] if len(hist) >= 5 else np.nan  # Q_{t-5}

        if pd.isna(q_last4): continue

        if len(hist) < 16: # Seasonal random walk (10-15 quarters of history)
            forecast = q_last4
            errors = hist['epspxq'].values[4:] - hist['epspxq'].values[:-4]
            sigma = errors.std(ddof=1)

        else: # If 16+quarters, Foster (1977): Q_t - Q_{t-4} = delta + phi*(Q_{t-1} - Q_{t-5})
            reg = hist[['y_foster', 'x_foster', 'pidx', 'pidx_l1', 'pidx_l4', 'pidx_l5']].dropna()
            reg = reg[
                (reg['pidx'] - reg['pidx_l1'] == 1) &
                (reg['pidx'] - reg['pidx_l4'] == 4) &
                (reg['pidx'] - reg['pidx_l5'] == 5)
            ]
            if len(reg) < 4: continue

            y = reg['y_foster'].values
            X = np.column_stack([np.ones(len(y)), reg['x_foster'].values])
            coefs, *_ = np.linalg.lstsq(X, y, rcond=None)
            delta, phi = coefs

            if pd.isna(q_last5):continue

            forecast  = q_last4 + delta + phi * (q_last - q_last5)
            residuals = y - X @ coefs
            sigma     = residuals.std(ddof=2)

        if sigma == 0 or pd.isna(sigma): continue

        # SUE = (actual - forecast) / sigma
        results.append({
            'gvkey'   : gvkey,
            'datadate': event_row['datadate'],
            'fyearq'  : event_row['fyearq'],
            'fqtr'    : event_row['fqtr'],
            'epspxq'  : event_row['epspxq'],
            'forecast': forecast,
            'sigma'   : sigma,
            'SUE'     : (event_row['epspxq'] - forecast) / sigma
        })

df_sue = pd.DataFrame(results)
print(f"{len(df_sue)} SUE observations computed")
if len(df_sue) > 0:
    print(df_sue['SUE'].describe())


103386 SUE observations computed
count    103386.000000
mean         -0.071095
std           3.340572
min        -637.009385
25%          -0.654765
50%           0.029560
75%           0.645079
max         253.717019
Name: SUE, dtype: float64


We merge this dataset with filtered firm-quarters to filter out incorrect quarters.

In [ ]:
df_sue['gvkey'] = df_sue['gvkey'].astype(str)
df_sue['datadate'] = pd.to_datetime(df_sue['datadate'])

dfQ_final['gvkey'] = dfQ_final['gvkey'].astype(str)
dfQ_final['datadate'] = pd.to_datetime(dfQ_final['datadate'])

df_sue_final = df_sue.merge(
    dfQ_final[['gvkey', 'datadate']],
    on=['gvkey', 'datadate'],
    how='inner'
)

df_sue_final.to_csv("data/SUE.csv", index=False)
print(f"{len(df_sue_final)} SUE after all filters")

84312 SUE after all filters


We have 84312 quarters: a bit less than expected, this might be because of diluted/basic EPS.

### **3.2.3. Portfolio assignment - hindisght bias**
"*Like FOS, we overcome that bias by assigning firms to portfolios on the basis of their standings relative to the distribution of unexpected earnings in the prior quarter*".

### **3.2.4. Continuously balanced SUE**

"*The continuously balanced SUE strategy works as follows. On a given   trading day, we identify any firms that announced earnings, and where standardized unexpected earnings fall in the upper quintile (good news) or lower quintile (bad news) of the prior-quarter distribution. If both good news and bad news firms exist for that day, we assume a long position in the former and a short position in the latter. The long (short) positions are initially equally weighted across the available good (bad) news firms, with the total amount of the long position exactly offsetting the total amount of the short position. We then compute buy-and-hold (i.e., continuously compounded) returns on each of the stocks in the long and short position, over the 60 trading days subsequent to the earnings announcement*."



"*On 14% of trading days, there were either no new good news or no bad news firms available, and so no match could be created. In such cases, we "wait" until a match becomes available. For example, if two good news firms announced earnings on day 1, but no bad news firms announced, we would wait until at least one bad news firm announced earnings. If the first available bad news firm announced on day 4, it would be matched with all good news firms announcing from days 1 through 4, and we would then compound returns from day 5 through day 64.In 97% of all cases, a match became available within two days*."

"*To provide some control for the Banz-Reinganum size effect, this matching process was always conducted within groups of small, medium, and large firms. Small firms are those whose January 1 market value of equity was among the lowest four deciles of the NYSE/AMEX, whereas large firms are those among the highest three deciles. Using only three size groups increased the probability of finding matches of good news and bad news firms within a short period of time. Since we used only three size groups (versus ten in the FOS control portfolio approach), our control for size is not as precise. However, if we assume that smaller firms are as likely to announce bad news as good news, this introduces no bias in the results.*"

**Steps**
- For every day, identify shortlisted firms that announced earnings
- Distribute prio-quarter SUE of every firms in quintiles
- We only keep firms for which SUE fell in the Q1 or Q5


If both Q1 and Q5 firms exist on that day:
- we assume a long-short position. Each leg of the position is equally weighted across available firms, with the long/short side perfectly offsetting each other.
- we compute continuously compounded buy&hold returns of each of the stocks of the position over the 60 days subsequent to the earnings announcement, without rebalancing

If both Q1 and Q5 firms aren't available, we wait until a match becomes available:
- If the first available bad news firm announced on day 4, it would be matched with all good news firms announcing from days 1 through 4, and we would then compound returns from day 5 through day 64
In 97% of all cases, a match became available within two days.

We begin by identyfing relevant Q1/Q5 quarters.




In [ ]:
# We identify prior quarter for each firm-quarter
df_sue_final['prev_fyearq'] = np.where(df_sue_final['fqtr'] == 1, df_sue_final['fyearq'] - 1, df_sue_final['fyearq'])
df_sue_final['prev_fqtr']   = np.where(df_sue_final['fqtr'] == 1, 4, df_sue_final['fqtr'] - 1)

# We compute SUE quintile boundaries from prior-quarter distribution
quintile_bounds = (
    df_sue_final.groupby(['fyearq', 'fqtr'])['SUE']
    .quantile([0.2, 0.8])
    .unstack()
    .reset_index()
    .rename(columns={0.2: 'Q20', 0.8: 'Q80', 'fyearq': 'prev_fyearq', 'fqtr': 'prev_fqtr'})
)

# We merge bounds onto each firm-quarter via its prior quarter
df_sue_final = df_sue_final.merge(quintile_bounds, on=['prev_fyearq', 'prev_fqtr'], how='left')

# We assign quintile label — only Q1 (bad news) and Q5 (good news) are kept
df_sue_final['SUE_quintile'] = np.where(df_sue_final['SUE'] <= df_sue_final['Q20'], 1,
                               np.where(df_sue_final['SUE'] >= df_sue_final['Q80'], 5, np.nan))

before = len(df_sue_final)
df_sue_final = df_sue_final[df_sue_final['SUE_quintile'].notna()]
print(f"Kept {len(df_sue_final)} firm-quarters in Q1 or Q5 (dropped {before - len(df_sue_final)})")
print(df_sue_final['SUE_quintile'].value_counts().sort_index())

df_sue_final = df_sue_final.merge(
    dfQ_final[['gvkey', 'datadate', 'PERMNO', 'rdq']],
    on=['gvkey', 'datadate'],
    how='left'
)

Kept 33650 firm-quarters in Q1 or Q5 (dropped 50662)
SUE_quintile
1.0    16752
5.0    16898
Name: count, dtype: int64


We then add the long/short mechanism, with the matching logic.

In [69]:
# We get sorted list of unique trading days
trading_days_list = sorted(dfD['DlyCalDt'].unique())
day_to_idx = {day: idx for idx, day in enumerate(trading_days_list)}  

# For each trading day, which firms announced?
announcements = df_sue_final[['gvkey', 'PERMNO', 'rdq', 'SUE_quintile']].copy()
announcements['rdq'] = pd.to_datetime(announcements['rdq'])

# Pedning firms waiting for a match
results_strategy, pending_good, pending_bad, wait_days_list = [], [], [], []
days_with_announcements, days_no_match = 0, 0
pending_since = None

for day in trading_days_list:
    today = announcements[announcements['rdq'] == day]
    new_good = today[today['SUE_quintile'] == 5][['PERMNO', 'rdq']].to_dict('records')
    new_bad  = today[today['SUE_quintile'] == 1][['PERMNO', 'rdq']].to_dict('records')

    had_announcement = len(new_good) > 0 or len(new_bad) > 0 # To compute % of interest

    pending_good.extend(new_good)
    pending_bad.extend(new_bad)

    # Track when waiting started
    if had_announcement and pending_since is None:
        pending_since = day

    # No match possible
    if not pending_good or not pending_bad:
        if had_announcement:
            days_no_match += 1
        if had_announcement:
            days_with_announcements += 1
        continue

    days_with_announcements += 1

    # Compute wait duration
    if pending_since is not None and pending_since != day:
        wait = day_to_idx[day] - day_to_idx[pending_since]
        wait_days_list.append(wait)
    else:
        wait_days_list.append(0)

    day_idx   = day_to_idx[day]
    start_day = trading_days_list[day_idx + 1]
    end_idx   = day_idx + 61
    if day_idx + 1 >= len(trading_days_list) or end_idx >= len(trading_days_list): continue
    end_day = trading_days_list[end_idx]

    results_strategy.append({
        'match_day' : day,
        'start_day' : start_day,
        'end_day'   : end_day,
        'good_firms': [f['PERMNO'] for f in pending_good],
        'bad_firms' : [f['PERMNO'] for f in pending_bad],
        'n_good'    : len(pending_good),
        'n_bad'     : len(pending_bad),
        'wait_days' : wait_days_list[-1],
    })

    pending_good, pending_bad = [], []
    pending_since = None

df_strategy = pd.DataFrame(results_strategy)
print(f"{len(df_strategy)} matched positions created")
print(df_strategy[['match_day', 'n_good', 'n_bad']].head(3))

2689 matched positions created
   match_day  n_good  n_bad
0 1974-04-26       5      1
1 1974-04-29       7      5
2 1974-04-30      12      5


We compute statistics of interest (14% and 97% in the corpus).

In [67]:
pct_no_match = days_no_match / days_with_announcements * 100 if days_with_announcements > 0 else 0
wait_series  = pd.Series(wait_days_list)
pct_within_2 = (wait_series <= 1).mean() * 100

print(f"Days with announcements : {days_with_announcements}")
print(f"Days with no match      : {days_no_match} ({pct_no_match:.1f}%)")
print(f"Wait days distribution  :\n{wait_series.value_counts().sort_index()}")
print(f"Match within 2 days     : {pct_within_2:.1f}%")

Days with announcements : 3057
Days with no match      : 368 (12.0%)
Wait days distribution  :
0    2388
1     236
2      47
3      14
4       1
5       1
7       1
8       1
Name: count, dtype: int64
Match within 2 days     : 97.6%


For days we no available match, we find 12% instead of the announced 14%. This can probably be fixed by reaching the correct amount of earnings announcement. For 2 days match, we find 97%, just like the paper.

From these matched positions, we can compute returns, with equally-weigthed positions offseting the initial value of the other leg. We normalize to $1 long / $1 short, equally weighted within each leg, with each good firm receiving 1/n_good, each bad firm getting 1/n_bad.
We vectorize it to treat our large dataset.

In [ ]:
# Pre-index dfD by PERMNO for fast lookup
dfD_indexed = dfD.set_index(['PERMNO', 'DlyCalDt']).sort_index()

# Explode df_strategy into one row per firm
long_rows = df_strategy.explode('good_firms').rename(columns={'good_firms': 'PERMNO'})
long_rows['leg'] = 'long'
short_rows = df_strategy.explode('bad_firms').rename(columns={'bad_firms': 'PERMNO'})
short_rows['leg'] = 'short'

positions = pd.concat([
    long_rows[['match_day', 'start_day', 'end_day', 'PERMNO', 'leg']],
    short_rows[['match_day', 'start_day', 'end_day', 'PERMNO', 'leg']]
]).reset_index(drop=True)

# Merge with dfD to get all daily returns for each position
positions['PERMNO'] = positions['PERMNO'].astype(dfD['PERMNO'].dtype)
dfD_small = dfD[['PERMNO', 'DlyCalDt', 'DlyReturns']].copy()

positions = positions.merge(dfD_small, on='PERMNO', how='left')
positions = positions[
    (positions['DlyCalDt'] >= positions['start_day']) &
    (positions['DlyCalDt'] <= positions['end_day'])
]

# Buy-and-hold return per firm per position
positions = positions.dropna(subset=['DlyReturns'])
bh_returns = (
    positions.groupby(['match_day', 'leg', 'PERMNO'])['DlyReturns']
    .apply(lambda x: (1 + x).prod() - 1)
    .reset_index()
    .rename(columns={'DlyReturns': 'bh_return'})
)

# Equally weighted per leg
leg_returns = (
    bh_returns.groupby(['match_day', 'leg'])['bh_return']
    .mean()
    .unstack()
    .reset_index()
    .rename(columns={'long': 'r_long', 'short': 'r_short'})
)

leg_returns['r_ls'] = leg_returns['r_long'] - leg_returns['r_short']

print(f"{len(leg_returns)} positions with computed returns")
print(leg_returns[['r_long', 'r_short', 'r_ls']].describe())
print(f"\nMean long-short return: {leg_returns['r_ls'].mean():.4f}")

2689 positions with computed returns
leg         r_long      r_short         r_ls
count  2689.000000  2689.000000  2689.000000
mean      0.140515     0.038981     0.101535
std       2.701230     0.377695     2.714439
min      -0.659091    -0.694805   -11.229967
25%      -0.045181    -0.067902    -0.051577
50%       0.039644     0.010974     0.031850
75%       0.138778     0.097100     0.115173
max     130.746257    11.284106   130.326753

Mean long-short return: 0.1015
